# Temporal dynamics — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## How does yearly volume change with data coverage?

In [ ]:
t=summary(['year']).sort_values('year');w=t.loc[t.units.idxmax()]
note(f"Il {int(w.year)} ha più unità: {units(w.units)}; il 2020 copre {df[df.year==2020].state.nunique()} stati contro {df[df.year==2021].state.nunique()} nel 2021.")
twobars(t.assign(year=t.year.astype(str)),'year',['units','revenue'],['Unità vendute per anno','Ricavi registrati per anno (USD)'])
data_table(t)
caution('Le differenze tra anni NON dimostrano crescita commerciale a parità di perimetro.')

## Which months have the highest and lowest recorded volume?

In [ ]:
t=summary(['year_month']).sort_values('year_month');a=t.loc[t.units.idxmax()];b=t.loc[t.units.idxmin()]
note(f"Picco unità: {a.year_month} ({units(a.units)}); minimo: {b.year_month} ({units(b.units)}). La serie mostra anche le differenze di copertura dei dati.")
lineplot(t,'year_month','units','Serie mensile: unità vendute')
data_table(t)

## Which weekdays contribute the most sales?

In [ ]:
t=summary(['weekday_number','weekday_name_it']).sort_values('weekday_number');t['units_per_observed_day']=t.units/t.days
m=t.loc[t.units.idxmax()]
note(f"Più unità aggregate: {m.weekday_name_it} ({units(m.units)}); il confronto giornaliero va letto insieme alla media per data osservata.")
twobars(t,'weekday_name_it',['units','units_per_observed_day'],['Unità totali per giorno settimana','Unità / data osservata'])
data_table(t)

## How do weekend and weekday averages compare?

In [ ]:
t=summary(['is_weekend']);t['period']=np.where(t.is_weekend,'Weekend','Feriale')
t['units_per_day']=t.units/t.days;t['revenue_per_day']=t.revenue/t.days;t['profit_per_day']=t.profit/t.days
r=t.loc[t.units_per_day.idxmax()]
note(f"Per data osservata, {r.period} ha più unità ({units(r.units_per_day)}). Confronto normalizzato sui giorni realmente presenti, non sulle righe.")
twobars(t,'period',['units_per_day','revenue_per_day'],['Unità / data osservata','Ricavi / data osservata'])
data_table(t)
caution('Una data presente non equivale a copertura omogenea di retailer e negozi.')

## Which seasons stand out within each year?

In [ ]:
t=summary(['year','season_it']);a=t.loc[t.units.idxmax()]
note(f"Maggiore combinazione anno-stagione per unità: {int(a.year)} – {a.season_it}, {units(a.units)}. L’inverno attraversa due anni civili.")
heat(t.assign(year=t.year.astype(str)),'season_it','year','units','Volumi per stagione e anno')
data_table(t)

## Which segments drive observed differences between years?

In [ ]:
parts=[]
for col in ['retailer','product_category','sales_method']:
    t=summary(['year',col]);pv=t.pivot_table(index=col,columns='year',values='units',fill_value=0)
    for y in [2020,2021]:
        if y not in pv:pv[y]=0
    pv['delta']=pv[2021]-pv[2020];pv['segmento']=col;pv['nome']=pv.index.astype(str);parts.append(pv.reset_index(drop=True))
r=pd.concat(parts,ignore_index=True);w=r.sort_values('delta',ascending=False).iloc[0]
note(f"Tra retailer/categoria/canale, maggiore variazione assoluta di unità: {w.segmento} = {w.nome}, {units(w.delta)}. Differenze descrittive, non crescita comparabile.")
bar(r.assign(label=r.segmento+': '+r.nome),'label','delta','Differenza osservata unità (2021 meno 2020)',top=17);plt.show()
data_table(r.sort_values('delta',ascending=False))
caution('Gli stati coperti aumentano da 9 a 46: confronto 2020/2021 non a perimetro costante.')